In [8]:
import time
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image
from datetime import timedelta
from tqdm import tqdm
import gc
import torchvision.ops as ops
from concurrent.futures import ThreadPoolExecutor

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })

    return tiles

# ======================== Overlay and Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    height, width, _ = img_shape

    if results.masks is None:
        return object_data

    for mask, box, cls, conf in zip(results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_layer = np.zeros((height, width), dtype=np.uint8)
        if mask is not None and len(mask) > 0:
            cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)

        pixel_count = np.count_nonzero(mask_layer[y1:y2, x1:x2])
        area = pixel_count * (transform.a * abs(transform.e))

        lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "area": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image
        })

    return object_data

# ======================== Utilities ========================
def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)

def preprocess_batch(batch):
    pil_imgs_batch = []
    rgb_shapes = []
    for slice_data in batch:
        img_array = slice_data["img_array"]
        rgb_img = np.moveaxis(img_array[:3], 0, -1)
        rgb_shapes.append(rgb_img.shape)
        pil_imgs_batch.append(Image.fromarray(rgb_img))
    return pil_imgs_batch, rgb_shapes

# ======================== Tile-Level Suppression ========================
def maximum_likelihood_suppression(results):
    if results.masks is None or results.boxes is None:
        return results

    conf = results.boxes.conf
    cls = results.boxes.cls

    selected_indices = []
    seen_classes = set()
    sorted_indices = torch.argsort(conf, descending=True)

    for idx in sorted_indices:
        class_id = int(cls[idx].item())
        if class_id not in seen_classes:
            selected_indices.append(idx)
            seen_classes.add(class_id)

    selected_indices = torch.tensor(selected_indices, dtype=torch.long)

    results.boxes.data = results.boxes.data[selected_indices]
    if results.masks is not None and results.masks.data is not None:
        results.masks.data = results.masks.data[selected_indices]

    return results

# ======================== Global Soft Suppression Across Tiles ========================
def global_mls_by_mask_iou(object_infos, iou_threshold=0.5):
    if not object_infos:
        return []

    df = pd.DataFrame(object_infos)
    retained_infos = []

    for class_id in df['class'].unique():
        group = df[df['class'] == class_id]

        if len(group) == 1:
            retained_infos.append(group.iloc[0].to_dict())
            continue

        boxes = []
        for _, row in group.iterrows():
            x = row['longitude']
            y = row['latitude']
            w = h = 0.99
            boxes.append([x, y, x + w, y + h])

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        scores_tensor = torch.tensor(group['confidence'].values, dtype=torch.float32)

        ious = ops.box_iou(boxes_tensor, boxes_tensor)
        keep = []
        suppressed = set()

        for i in range(len(scores_tensor)):
            if i in suppressed:
                continue
            keep.append(i)
            for j in range(i + 1, len(scores_tensor)):
                if ious[i, j] > iou_threshold:
                    scores_tensor[j] *= (1 - ious[i, j])  # soft suppression
                    if scores_tensor[j] < 0.1:
                        suppressed.add(j)

        retained_infos.extend(group.iloc[keep].to_dict(orient='records'))

    return retained_infos

# ======================== Load YOLO Model ========================
model = YOLO("C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

_ = model([Image.fromarray(np.zeros((3000, 3000, 3), dtype=np.uint8))], device=device, verbose=False)

# ======================== Parameters ========================
window_size = 3000
stride = 1500
batch_size = 32
folder_path = './geotiff_sample'

# ======================== Main Inference Loop ========================
tif_images = find_tif_images(folder_path)
accumulate_object_infos = []
total_images = len(tif_images)

overall_bar = tqdm(
    enumerate(tif_images, 1),
    total=total_images,
    desc="Overall Progress",
    unit="image"
)

for idx, tif_image in overall_bar:
    image_start_time = time.time()
    slices = slide_geotiff(tif_image, window_size=window_size, stride=stride)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]
    image_object_infos = []

    for batch in slice_batches:
        pil_imgs_batch, rgb_shapes = preprocess_batch(batch)
        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results, shape in zip(batch, results_batch, rgb_shapes):
            results = maximum_likelihood_suppression(results)
            object_info = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=shape,
                tif_image=os.path.basename(tif_image)
            )
            image_object_infos.extend(object_info)

    accumulate_object_infos.extend(global_mls_by_mask_iou(image_object_infos))
    torch.cuda.empty_cache()
    gc.collect()

    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(time.time() - image_start_time))}",
        'Images_left': total_images - idx,
    })

final_csv_path = "lat_cong_csv/temp.csv"
pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")


Using device: cuda


Overall Progress: 100%|███████████████████████| 4/4 [03:30<00:00, 52.61s/image, Last_image_time=0:00:55, Images_left=0]


✅ Final results saved to lat_cong_csv/temp.csv


In [21]:
import time
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image
from datetime import timedelta
from tqdm.notebook import tqdm
import gc

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })

    return tiles


# ======================== Overlay and Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    height, width, _ = img_shape

    if results.masks is None:
        return object_data

    for mask, box, cls, conf in zip(results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_layer = np.zeros((height, width), dtype=np.uint8)
        if mask is not None and len(mask) > 0:
            cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)

        pixel_count = np.count_nonzero(mask_layer[y1:y2, x1:x2])
        area = pixel_count * (transform.a * abs(transform.e))

        lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "area": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image
        })

    return object_data

# ======================== Utilities ========================
def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)


# ======================== Maximum Likelihood Suppression ========================
from ultralytics.engine.results import Results
from ultralytics.utils import ops

from torchvision.ops import box_iou
from ultralytics.engine.results import Results

def maximum_likelihood_suppression(results, iou_threshold=0.5, conf_threshold=0.6):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    boxes = results.boxes.xyxy  # shape: [N, 4]
    confs = results.boxes.conf  # shape: [N]
    classes = results.boxes.cls  # shape: [N]

    # Filter low-confidence predictions first
    keep = confs >= conf_threshold
    boxes = boxes[keep]
    confs = confs[keep]
    classes = classes[keep]

    if boxes.shape[0] == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks is not None and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    # Sort by descending confidence
    order = torch.argsort(confs, descending=True)
    boxes = boxes[order]
    confs = confs[order]
    classes = classes[order]
    if results.masks is not None and results.masks.data is not None:
        masks = results.masks.data[keep][order]
    else:
        masks = None

    keep_indices = []
    while boxes.size(0) > 0:
        curr_box = boxes[0].unsqueeze(0)
        curr_cls = int(classes[0].item())

        keep_indices.append(order[0].item())  # store original index before filtering

        if boxes.size(0) == 1:
            break

        # Remove all other boxes of same class with high IoU
        ious = box_iou(curr_box, boxes[1:])[0]
        class_mask = classes[1:] == curr_cls
        suppress = (ious > iou_threshold) & class_mask

        # Keep only boxes that are not suppressed
        keep_mask = ~suppress

        boxes = boxes[1:][keep_mask]
        confs = confs[1:][keep_mask]
        classes = classes[1:][keep_mask]
        order = order[1:][keep_mask]
        if masks is not None:
            masks = masks[1:][keep_mask]

    selected_indices = torch.tensor(keep_indices, dtype=torch.long)

    results.boxes.data = results.boxes.data[selected_indices]
    if results.masks is not None and results.masks.data is not None:
        results.masks.data = results.masks.data[selected_indices]

    return results



# ======================== Load YOLO Model ========================
model = YOLO("C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

# ======================== Parameters ========================
window_size = 3000
stride = 1500
batch_size = 32
# folder_path = 'E:/Data/al_30cm_2021_transfer_2_of_2/'
folder_path = './geotiff_sample'

# ======================== Main Inference Loop ========================
tif_images = find_tif_images(folder_path)
accumulate_object_infos = []
total_images = len(tif_images)

overall_bar = tqdm(
    enumerate(tif_images, 1), 
    total=total_images, 
    desc="Overall Progress",
    unit="image",
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
)

for idx, tif_image in overall_bar:
    image_start_time = time.time()
    slices = slide_geotiff(tif_image, window_size=window_size, stride=stride)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

    batch_bar = tqdm(
        slice_batches,
        desc=f"Processing {os.path.basename(tif_image)}",
        leave=False,
        unit="batch",
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )

    for batch in batch_bar:
        pil_imgs_batch = []
        for slice_data in batch:
            img_array = slice_data["img_array"]
            rgb_img = np.moveaxis(img_array[:3], 0, -1)
            pil_imgs_batch.append(Image.fromarray(rgb_img))

        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results in zip(batch, results_batch):
            results = maximum_likelihood_suppression(results)
            object_info = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=rgb_img.shape,
                tif_image=os.path.basename(tif_image)
            )
            accumulate_object_infos.extend(object_info)

    # Save intermediate results
    # intermediate_csv = f"data/sub_2/intermediate_object_info_{idx}.csv"
    # pd.DataFrame(accumulate_object_infos).to_csv(intermediate_csv, index=False)

    # Cleanup
    del slices, slice_batches, batch_bar, pil_imgs_batch, results_batch, img_array, rgb_img
    torch.cuda.empty_cache()
    gc.collect()

    image_time_taken = time.time() - image_start_time
    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(image_time_taken))}",
        'Images_left': total_images - idx,
    })

overall_bar.close()

final_csv_path = "lat_cong_csv/temp.csv"
pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")

Using device: cuda


Overall Progress:   0%|          | 0/4 [00:00<?, ?image/s]

Processing m_3108501_ne_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?batch/s]

Processing m_3108501_nw_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?batch/s]

Processing m_3108501_se_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?batch/s]

Processing m_3108506_se_16_030_20211113.tif:   0%|          | 0/7 [00:00<?, ?batch/s]


✅ Final results saved to lat_cong_csv/temp.csv


In [27]:
import time
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image
from datetime import timedelta
from tqdm.notebook import tqdm
import gc

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })

    return tiles


# ======================== Overlay and Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    height, width, _ = img_shape

    if results.masks is None:
        return object_data

    for mask, box, cls, conf in zip(results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_layer = np.zeros((height, width), dtype=np.uint8)
        if mask is not None and len(mask) > 0:
            cv2.fillPoly(mask_layer, [np.array(mask, dtype=np.int32)], 255)

        pixel_count = np.count_nonzero(mask_layer[y1:y2, x1:x2])
        area = pixel_count * (transform.a * abs(transform.e))

        lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "area": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image
        })

    return object_data

# ======================== Utilities ========================
def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)


# ======================== Maximum Likelihood Suppression ========================
from ultralytics.engine.results import Results
from ultralytics.utils import ops

from torchvision.ops import box_iou
from ultralytics.engine.results import Results

from torchvision.ops import box_iou

def maximum_likelihood_suppression(results, iou_threshold=0.6, conf_threshold=0.6):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    boxes = results.boxes.xyxy  # [N, 4]
    confs = results.boxes.conf
    classes = results.boxes.cls

    keep = confs >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    boxes = boxes[keep]
    confs = confs[keep]
    classes = classes[keep]
    orig_indices = torch.arange(len(results.boxes), device=keep.device)[keep]

    if results.masks and results.masks.data is not None:
        masks = results.masks.data[keep]
    else:
        masks = None

    keep_indices = []

    for cls_id in torch.unique(classes).tolist():
        cls_mask = classes == cls_id
        cls_boxes = boxes[cls_mask]
        cls_confs = confs[cls_mask]
        cls_indices = orig_indices[cls_mask]
        if masks is not None:
            cls_masks = masks[cls_mask]
        else:
            cls_masks = None

        if cls_boxes.size(0) == 0:
            continue

        order = torch.argsort(cls_confs, descending=True)
        cls_boxes = cls_boxes[order]
        cls_confs = cls_confs[order]
        cls_indices = cls_indices[order]
        if cls_masks is not None:
            cls_masks = cls_masks[order]

        suppressed = torch.zeros(cls_boxes.size(0), dtype=torch.bool, device=cls_boxes.device)

        for i in range(cls_boxes.size(0)):
            if suppressed[i]:
                continue
            keep_indices.append(cls_indices[i].item())
            ious = box_iou(cls_boxes[i].unsqueeze(0), cls_boxes[i+1:])[0]
            suppress_mask = ious > iou_threshold
            suppressed[i+1:] |= suppress_mask

    selected_indices = torch.tensor(keep_indices, dtype=torch.long, device=results.boxes.data.device)

    results.boxes.data = results.boxes.data[selected_indices]
    if results.masks is not None and results.masks.data is not None:
        results.masks.data = results.masks.data[selected_indices]

    return results



# ======================== Load YOLO Model ========================
model = YOLO("C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

# ======================== Parameters ========================
window_size = 3000
stride = 1500
batch_size = 32
# folder_path = 'E:/Data/al_30cm_2021_transfer_2_of_2/'
folder_path = './geotiff_sample'

# ======================== Main Inference Loop ========================
tif_images = find_tif_images(folder_path)
accumulate_object_infos = []
total_images = len(tif_images)

overall_bar = tqdm(
    enumerate(tif_images, 1), 
    total=total_images, 
    desc="Overall Progress",
    unit="image",
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
)

for idx, tif_image in overall_bar:
    image_start_time = time.time()
    slices = slide_geotiff(tif_image, window_size=window_size, stride=stride)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

    batch_bar = tqdm(
        slice_batches,
        desc=f"Processing {os.path.basename(tif_image)}",
        leave=False,
        unit="batch",
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )

    for batch in batch_bar:
        pil_imgs_batch = []
        for slice_data in batch:
            img_array = slice_data["img_array"]
            rgb_img = np.moveaxis(img_array[:3], 0, -1)
            pil_imgs_batch.append(Image.fromarray(rgb_img))

        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results in zip(batch, results_batch):
            results = maximum_likelihood_suppression(results)
            object_info = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=rgb_img.shape,
                tif_image=os.path.basename(tif_image)
            )
            accumulate_object_infos.extend(object_info)

    # Save intermediate results
    # intermediate_csv = f"data/sub_2/intermediate_object_info_{idx}.csv"
    # pd.DataFrame(accumulate_object_infos).to_csv(intermediate_csv, index=False)

    # Cleanup
    del slices, slice_batches, batch_bar, pil_imgs_batch, results_batch, img_array, rgb_img
    torch.cuda.empty_cache()
    gc.collect()

    image_time_taken = time.time() - image_start_time
    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(image_time_taken))}",
        'Images_left': total_images - idx,
    })

overall_bar.close()

final_csv_path = "lat_cong_csv/temp.csv"
pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")

Using device: cuda


Overall Progress:   0%|          | 0/4 [00:00<?, ?image/s]

Processing m_3108501_ne_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?batch/s]

Processing m_3108501_nw_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?batch/s]

Processing m_3108501_se_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?batch/s]

Processing m_3108506_se_16_030_20211113.tif:   0%|          | 0/7 [00:00<?, ?batch/s]


✅ Final results saved to lat_cong_csv/temp.csv


In [28]:
import time
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image
from datetime import timedelta
from tqdm.notebook import tqdm
import gc

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })

    return tiles

# ======================== Overlay and Extraction with Mask Return ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    masks_out = []

    height, width, _ = img_shape
    if results.masks is None:
        return object_data, masks_out

    for mask, box, cls, conf in zip(results.masks.data, results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_np = mask.cpu().numpy().astype(np.uint8)
        pixel_count = np.count_nonzero(mask_np)
        area = pixel_count * (transform.a * abs(transform.e))
        lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "area": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image
        })

        masks_out.append(mask_np)

    return object_data, masks_out

# ======================== Utilities ========================
def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)

# ======================== Local NMS ========================
from torchvision.ops import box_iou
def maximum_likelihood_suppression(results, iou_threshold=0.5, conf_threshold=0.6):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    boxes = results.boxes.xyxy
    confs = results.boxes.conf
    classes = results.boxes.cls

    keep = confs >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    boxes = boxes[keep]
    confs = confs[keep]
    classes = classes[keep]
    orig_indices = torch.arange(len(results.boxes), device=keep.device)[keep]

    masks = results.masks.data[keep] if results.masks and results.masks.data is not None else None
    keep_indices = []

    for cls_id in torch.unique(classes).tolist():
        cls_mask = classes == cls_id
        cls_boxes = boxes[cls_mask]
        cls_confs = confs[cls_mask]
        cls_indices = orig_indices[cls_mask]
        cls_masks = masks[cls_mask] if masks is not None else None

        if cls_boxes.size(0) == 0:
            continue

        order = torch.argsort(cls_confs, descending=True)
        cls_boxes = cls_boxes[order]
        cls_confs = cls_confs[order]
        cls_indices = cls_indices[order]
        cls_masks = cls_masks[order] if cls_masks is not None else None

        suppressed = torch.zeros(cls_boxes.size(0), dtype=torch.bool, device=cls_boxes.device)

        for i in range(cls_boxes.size(0)):
            if suppressed[i]:
                continue
            keep_indices.append(cls_indices[i].item())
            ious = box_iou(cls_boxes[i].unsqueeze(0), cls_boxes[i+1:])[0]
            suppressed[i+1:] |= (ious > iou_threshold)

    selected_indices = torch.tensor(keep_indices, dtype=torch.long, device=results.boxes.data.device)
    results.boxes.data = results.boxes.data[selected_indices]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[selected_indices]

    return results

# ======================== Global Mask Suppression ========================
def compute_iou_mask(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0.0

def global_mls_by_mask_iou(object_infos, masks, iou_thresh=0.5):
    keep_indices = []
    suppressed = [False] * len(object_infos)

    for i in range(len(object_infos)):
        if suppressed[i]:
            continue
        keep_indices.append(i)
        for j in range(i + 1, len(object_infos)):
            if suppressed[j]:
                continue
            if object_infos[i]["class"] == object_infos[j]["class"]:
                iou = compute_iou_mask(masks[i], masks[j])
                if iou > iou_thresh:
                    if object_infos[j]["confidence"] < object_infos[i]["confidence"]:
                        suppressed[j] = True
                    else:
                        suppressed[i] = True
                        keep_indices.pop()
                        break

    return [object_infos[i] for i in keep_indices]

# ======================== Load Model ========================
model = YOLO("C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

# ======================== Params ========================
window_size = 3000
stride = 1500
batch_size = 32
folder_path = './geotiff_sample'
final_csv_path = "lat_cong_csv/temp1.csv"

# ======================== Inference ========================
tif_images = find_tif_images(folder_path)
accumulate_object_infos = []
accumulate_masks = []
total_images = len(tif_images)

overall_bar = tqdm(enumerate(tif_images, 1), total=total_images, desc="Overall Progress", unit="image")

for idx, tif_image in overall_bar:
    start_time = time.time()
    slices = slide_geotiff(tif_image, window_size, stride)
    slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

    for batch in tqdm(slice_batches, desc=f"Processing {os.path.basename(tif_image)}", leave=False):
        pil_imgs_batch = [Image.fromarray(np.moveaxis(slice_data["img_array"][:3], 0, -1)) for slice_data in batch]
        results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

        for slice_data, results in zip(batch, results_batch):
            results = maximum_likelihood_suppression(results)
            object_info, masks = overlay_segmentation_on_array(
                results,
                transform=slice_data["transform"],
                img_shape=pil_imgs_batch[0].size + (3,),
                tif_image=os.path.basename(tif_image)
            )
            accumulate_object_infos.extend(object_info)
            accumulate_masks.extend(masks)

    del slices, slice_batches, results_batch, pil_imgs_batch
    torch.cuda.empty_cache()
    gc.collect()

    elapsed = time.time() - start_time
    overall_bar.set_postfix({
        'Last_image_time': f"{timedelta(seconds=int(elapsed))}",
        'Images_left': total_images - idx,
    })

overall_bar.close()

# ======================== Final Deduplication and Save ========================
filtered_infos = global_mls_by_mask_iou(accumulate_object_infos, accumulate_masks, iou_thresh=0.5)
pd.DataFrame(filtered_infos).to_csv(final_csv_path, index=False)
print(f"\n✅ Final results saved to {final_csv_path}")

Using device: cuda


Overall Progress:   0%|          | 0/4 [00:00<?, ?image/s]

Processing m_3108501_ne_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108501_nw_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108501_se_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108506_se_16_030_20211113.tif:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Final results saved to lat_cong_csv/temp1.csv


In [40]:
import time
import os
import glob
import numpy as np
import pandas as pd
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from PIL import Image
from datetime import timedelta
from tqdm.notebook import tqdm
import gc
from ultralytics import YOLO
from sklearn.cluster import DBSCAN
from torchvision.ops import box_iou

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x)
                })

    return tiles

# ======================== Extract Object Info + Masks ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image):
    object_data = []
    masks_out = []

    height, width, _ = img_shape
    if results.masks is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(results.masks.data, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_np = mask.cpu().numpy().astype(np.uint8)
        pixel_count = np.count_nonzero(mask_np)
        area = pixel_count * (transform.a * abs(transform.e))
        lon, lat = transform * ((x1 + x2) / 2, (y1 + y2) / 2)

        # Optional: also extract other fields from YOLO results if available
        box_info = {
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "width": x2 - x1,
            "height": y2 - y1
        }

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "x_center": (x1 + x2) / 2,
            "y_center": (y1 + y2) / 2,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image,
            "mask_shape": mask_np.shape,
            "mask_index": idx  # optional: useful for tracing back mask
        })

        masks_out.append(mask_np)

    return object_data, masks_out


# ======================== Per-tile Confidence Filtering ========================
def maximum_likelihood_suppression(results, iou_threshold=0.3, conf_threshold=0.7):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    boxes = results.boxes.xyxy
    confs = results.boxes.conf
    keep = confs >= conf_threshold

    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    results.boxes.data = results.boxes.data[keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[keep]
    return results

# ======================== Global Clustering Suppression ========================
def compute_mask_iou(mask1, mask2):
    inter = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return inter / union if union > 0 else 0.0

def get_mask_centroid(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return np.array([0.0, 0.0])
    return np.array([np.mean(xs), np.mean(ys)])

def suppress_duplicates_global(object_infos, masks, eps=30, iou_thresh=0.3):
    centroids = np.array([get_mask_centroid(m) for m in masks])
    clustering = DBSCAN(eps=eps, min_samples=1).fit(centroids)
    labels = clustering.labels_

    final_objects = []
    for cluster_id in np.unique(labels):
        idxs = np.where(labels == cluster_id)[0]
        if len(idxs) == 1:
            final_objects.append(object_infos[idxs[0]])
            continue

        keep_idx = idxs[0]
        for i in idxs[1:]:
            iou = compute_mask_iou(masks[keep_idx], masks[i])
            if iou > iou_thresh:
                if object_infos[i]['confidence'] > object_infos[keep_idx]['confidence']:
                    keep_idx = i
        final_objects.append(object_infos[keep_idx])
    return final_objects

# ======================== Inference Pipeline ========================
def run_inference(folder_path, model_path, final_csv_path,
                  window_size=3000, stride=1500, batch_size=32):

    model = YOLO(model_path)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    accumulate_object_infos, accumulate_masks = [], []
    total_images = len(tif_images)

    overall_bar = tqdm(enumerate(tif_images, 1), total=total_images, desc="Overall Progress", unit="image")

    for idx, tif_image in overall_bar:
        start_time = time.time()
        slices = slide_geotiff(tif_image, window_size, stride)
        slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

        for batch in tqdm(slice_batches, desc=f"Processing {os.path.basename(tif_image)}", leave=False):
            pil_imgs_batch = [Image.fromarray(np.moveaxis(slice_data["img_array"][:3], 0, -1)) for slice_data in batch]
            results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results, conf_threshold=0.7)
                object_info, masks = overlay_segmentation_on_array(
                    results,
                    transform=slice_data["transform"],
                    img_shape=pil_imgs_batch[0].size + (3,),
                    tif_image=os.path.basename(tif_image)
                )
                accumulate_object_infos.extend(object_info)
                accumulate_masks.extend(masks)

        torch.cuda.empty_cache()
        gc.collect()
        elapsed = time.time() - start_time
        overall_bar.set_postfix({
            'Last_image_time': f"{timedelta(seconds=int(elapsed))}",
            'Images_left': total_images - idx,
        })

    overall_bar.close()

    print(f"\n🔍 Suppressing duplicates across all tiles...")
    filtered_infos = suppress_duplicates_global(
        accumulate_object_infos, accumulate_masks,
        eps=21,          # centroid pixel distance threshold
        iou_thresh=0.5   # minimum mask IoU for duplicate suppression
    )

    pd.DataFrame(filtered_infos).to_csv(final_csv_path, index=False)
    print(f"\n✅ Final results saved to {final_csv_path}")

# ======================== Run Inference ========================
run_inference(
    folder_path="./geotiff_sample",
    model_path="C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="lat_cong_csv/temp.csv"
)


Using device: cuda


Overall Progress:   0%|          | 0/4 [00:00<?, ?image/s]

Processing m_3108501_ne_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108501_nw_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108501_se_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108506_se_16_030_20211113.tif:   0%|          | 0/7 [00:00<?, ?it/s]


🔍 Suppressing duplicates across all tiles...

✅ Final results saved to lat_cong_csv/temp.csv


In [46]:
import os
import glob
import time
import gc
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from itertools import combinations
from collections import defaultdict

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })
    return tiles

# ======================== Mask IOU + Global Centroid ========================
def compute_mask_iou(mask1, mask2):
    inter = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return inter / union if union > 0 else 0.0

def get_global_centroid(obj):
    return np.array([obj['global_x'], obj['global_y']])

# ======================== Overlay + Extract ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(results.masks.data, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):
        cls = int(cls.item())
        x1, y1, x2, y2 = map(int, box.tolist())

        mask_np = mask.cpu().numpy().astype(np.uint8)
        pixel_count = np.count_nonzero(mask_np)
        area = pixel_count * (transform.a * abs(transform.e))

        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy

        lon, lat = rasterio.transform.xy(transform, local_cy, local_cx)

        object_data.append({
            "latitude": lat,
            "longitude": lon,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": float(conf.item()),
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_shape": mask_np.shape
        })

        masks_out.append(mask_np)
    return object_data, masks_out

# ======================== Per-Tile Confidence Filter ========================
def maximum_likelihood_suppression(results, conf_threshold=0.3):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    confs = results.boxes.conf
    keep = confs >= conf_threshold

    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    results.boxes.data = results.boxes.data[keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[keep]
    return results

# ======================== Final Fusion Fix ========================
def fuse_duplicates_global(object_infos, masks, iou_thresh=0.5, dist_thresh=60):
    coord_bins = defaultdict(list)
    for i, obj in enumerate(object_infos):
        key = (int(obj['global_x'] // 25), int(obj['global_y'] // 25))  # bin size
        coord_bins[key].append(i)

    fused = []
    for bin_group in coord_bins.values():
        clustered = [-1] * len(bin_group)
        group_id = 0

        for i_idx, i in enumerate(bin_group):
            if clustered[i_idx] != -1:
                continue
            clustered[i_idx] = group_id
            for j_idx in range(i_idx + 1, len(bin_group)):
                j = bin_group[j_idx]
                if clustered[j_idx] != -1:
                    continue
                dist = np.linalg.norm(get_global_centroid(object_infos[i]) - get_global_centroid(object_infos[j]))
                iou = compute_mask_iou(masks[i], masks[j])
                if dist < dist_thresh and iou > iou_thresh:
                    clustered[j_idx] = group_id
            group_id += 1

        for group in range(group_id):
            idxs = [bin_group[i] for i, c in enumerate(clustered) if c == group]
            objs = [object_infos[i] for i in idxs]
            fused_obj = {
                "latitude": np.mean([o["latitude"] for o in objs]),
                "longitude": np.mean([o["longitude"] for o in objs]),
                "global_x": np.mean([o["global_x"] for o in objs]),
                "global_y": np.mean([o["global_y"] for o in objs]),
                "x_center": np.mean([o["x_center"] for o in objs]),
                "y_center": np.mean([o["y_center"] for o in objs]),
                "bbox_x1": np.mean([o["bbox_x1"] for o in objs]),
                "bbox_y1": np.mean([o["bbox_y1"] for o in objs]),
                "bbox_x2": np.mean([o["bbox_x2"] for o in objs]),
                "bbox_y2": np.mean([o["bbox_y2"] for o in objs]),
                "bbox_width": np.mean([o["bbox_width"] for o in objs]),
                "bbox_height": np.mean([o["bbox_height"] for o in objs]),
                "area_geo": np.mean([o["area_geo"] for o in objs]),
                "pixel_count": int(np.mean([o["pixel_count"] for o in objs])),
                "class": objs[0]["class"],
                "confidence": max(o["confidence"] for o in objs),
                "tile_name": objs[0]["tile_name"],
                "mask_index": objs[0]["mask_index"],
                "mask_shape": objs[0]["mask_shape"]
            }
            fused.append(fused_obj)

    return fused

# ======================== Inference Pipeline ========================
def run_inference(folder_path, model_path, final_csv_path,
                  window_size=3000, stride=1500, batch_size=32):

    model = YOLO(model_path)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    accumulate_object_infos, accumulate_masks = [], []
    total_images = len(tif_images)

    overall_bar = tqdm(enumerate(tif_images, 1), total=total_images, desc="Overall Progress", unit="image")

    for idx, tif_image in overall_bar:
        start_time = time.time()
        slices = slide_geotiff(tif_image, window_size, stride)
        slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

        for batch in tqdm(slice_batches, desc=f"Processing {os.path.basename(tif_image)}", leave=False):
            pil_imgs_batch = [Image.fromarray(np.moveaxis(slice_data["img_array"][:3], 0, -1)) for slice_data in batch]
            results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results, conf_threshold=0.3)
                object_info, masks = overlay_segmentation_on_array(
                    results,
                    transform=slice_data["transform"],
                    img_shape=pil_imgs_batch[0].size + (3,),
                    tif_image=os.path.basename(tif_image),
                    offset_xy=slice_data["offset_xy"]
                )
                accumulate_object_infos.extend(object_info)
                accumulate_masks.extend(masks)

        torch.cuda.empty_cache()
        gc.collect()
        elapsed = time.time() - start_time
        overall_bar.set_postfix({
            'Last_image_time': f"{timedelta(seconds=int(elapsed))}",
            'Images_left': total_images - idx,
        })

    overall_bar.close()

    print(f"\n🔍 Fusing overlapping detections across tiles...")
    filtered_infos = fuse_duplicates_global(
        accumulate_object_infos, accumulate_masks,
        iou_thresh=0.5,
        dist_thresh=60
    )

    pd.DataFrame(filtered_infos).to_csv(final_csv_path, index=False)
    print(f"\n✅ Final results saved to {final_csv_path}")

# ======================== Run It ========================
run_inference(
    folder_path="./geotiff_sample",
    model_path="C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="lat_cong_csv/final_results.csv"
)


Using device: cuda


Overall Progress:   0%|          | 0/4 [00:00<?, ?image/s]

Processing m_3108501_ne_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108501_nw_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108501_se_16_030_20211103.tif:   0%|          | 0/7 [00:00<?, ?it/s]

Processing m_3108506_se_16_030_20211113.tif:   0%|          | 0/7 [00:00<?, ?it/s]


🔍 Fusing overlapping detections across tiles...

✅ Final results saved to lat_cong_csv/final_results.csv


In [3]:
import os
import glob
import time
import gc
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window

# ======================== Sliding Window Reader ========================
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width
        transform, crs, dtype, count = src.transform, src.crs, src.dtypes[0], src.count

        for y in range(0, height - window_size + 1, stride):
            for x in range(0, width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)

                new_transform = from_origin(
                    transform.c + x * transform.a,
                    transform.f + y * transform.e,
                    transform.a,
                    -transform.e
                )

                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": crs,
                    "dtype": dtype,
                    "count": count,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })
    return tiles

# ======================== Overlay + GIS-Consistent Info ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        # 1. Original pixel mask polygon
        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)  # (N, 2)

        # 2. Convert pixel coordinates to GIS coordinates
        mask_gis_coords = []
        for pt in mask_pixel_coords:
            px, py = pt
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        # 3. Other metrics
        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))

        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy

        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, x2)

        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": str(mask_pixel_coords.tolist()),   # pixel
            "mask_points_gis": str(mask_gis_coords)           # GIS
        })

        masks_out.append(mask_pixel_coords)

    return object_data, masks_out


# ======================== Per-Tile Confidence Filter ========================
def maximum_likelihood_suppression(results, conf_threshold=0.5):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results

    confs = results.boxes.conf
    keep = confs >= conf_threshold

    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results

    results.boxes.data = results.boxes.data[keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[keep]
    return results

# ======================== Inference Runner ========================
def run_inference(folder_path, model_path, final_csv_path,
                  window_size=3000, stride=3000, batch_size=32):

    model = YOLO(model_path)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    accumulate_object_infos, accumulate_masks = [], []
    total_images = len(tif_images)

    overall_bar = tqdm(enumerate(tif_images, 1), total=total_images, desc="Overall Progress", unit="image")

    for idx, tif_image in overall_bar:
        start_time = time.time()
        slices = slide_geotiff(tif_image, window_size, stride)
        slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]

        for batch in tqdm(slice_batches, desc=f"Processing {os.path.basename(tif_image)}", leave=False):
            pil_imgs_batch = [Image.fromarray(np.moveaxis(slice_data["img_array"][:3], 0, -1)) for slice_data in batch]
            results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)

            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results, conf_threshold=0.5)
                object_info, masks = overlay_segmentation_on_array(
                    results,
                    transform=slice_data["transform"],
                    img_shape=pil_imgs_batch[0].size + (3,),
                    tif_image=os.path.basename(tif_image),
                    offset_xy=slice_data["offset_xy"]
                )
                accumulate_object_infos.extend(object_info)
                accumulate_masks.extend(masks)
        
        torch.cuda.empty_cache()
        gc.collect()
        elapsed = time.time() - start_time
        overall_bar.set_postfix({
            'Last_image_time': f"{timedelta(seconds=int(elapsed))}",
            'Images_left': total_images - idx,
        })

    overall_bar.close()

    pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
    print(f"\n✅ Final results saved to {final_csv_path}")


run_inference(
    folder_path="./geotiff_sample",
    model_path="C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="lat_cong_csv/final_yolo_nodedupe.csv"
)


Using device: cuda


Overall Progress:   0%|          | 0/4 [00:00<?, ?image/s]

Processing m_3108501_ne_16_030_20211103.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3108501_nw_16_030_20211103.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3108501_se_16_030_20211103.tif:   0%|          | 0/2 [00:00<?, ?it/s]

Processing m_3108506_se_16_030_20211113.tif:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Final results saved to lat_cong_csv/final_yolo_nodedupe.csv
